# Workplace choice

Zone of the usual workplace of a worker, given the home zone.

Article table: `est_jobchoice`. This notebook goes from the raw data to the estimates of this model:

1. Raw data
2. Data preparation
3. Estimation data
4. Descriptive statistics
5. Correlation of the explanatory variables (heatmaps, VIF)
6. Model specification (the folder's `model/estimate_workplace_choice.py`, unchanged)
7. Log-likelihood at the article coefficients
8. Biogeme estimation
9. Results: article format, fit, comparison with the article, correlation of the estimates
10. Hold-out sample

**Kernel:** select *Python (biogeme)* (`~/.venvs/biogeme`) in VS Code, then *Run All*. The switches in the next cell decide what is recomputed.

In [ ]:
RUN_DATA_PREP  = False   # True: rebuild the estimation tables from input/data/raw (section 2); a copy of the provided tables is kept first
RUN_CHECK      = True    # log-likelihood at the article coefficients
RUN_ESTIMATION = True    # Biogeme estimation, written to output/
RUN_HOLDOUT    = False    # log-likelihood of the hold-out sample at your estimates

import os, sys, json, glob, inspect, importlib, time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import biogeme.biogeme as bio
from biogeme.parameters import Parameters
from biogeme.results_processing import EstimationResults

FOLDER = os.path.abspath("")                       # this model folder (VS Code starts the notebook here)
sys.path.insert(0, os.path.join(FOLDER, "..", "..", "tools")); import nbtools as nt
sys.path.insert(0, os.path.join(FOLDER, "model"))
RUN = json.load(open(os.path.join(FOLDER, "RUN.json")))
OUT = os.path.join(FOLDER, "output"); os.makedirs(OUT, exist_ok=True)
NAME = "workplace_choice"
pd.set_option("display.max_rows", 200, "display.max_columns", 60, "display.width", 200)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("model folder:", FOLDER)

## 1. Raw data

Where each source comes from and where it goes (paths from `manual_estimation/`). *verify*: no download link is recorded in the project. The full download steps are in `INSTRUCTIONS.pdf`.

**Regional Travel Survey 2017/18 (household, person, trip records)** — Baltimore Metropolitan Council (BMC) and National Capital Region Transportation Planning Board (TPB/MWCOG)
- Where: <https://www.baltometro.org>
- How: Restricted: request the 2017/2018 Regional Travel Survey data from BMC / TPB. File used: RTS_raw_flat_data.csv (flat person-level file with household fields: household_id, person_id, home_state_county_fips, home_tract_fips, home_type, hhsize, numvehicle, numdrivers, hh_income_detailed, wthhfin, wtperfin, age, age_group, employment_status, j1_workplace_loc, work_state_county_fips, work_tract_fips, license, school locations and BMC/TPB TAZ codes). Also rts_trips_clean.csv (cleaned trips) for the synthetic-population workplace step.
- Put in: `00_shared_data/raw/rts_2017_18/`
- Files: `RTS_raw_flat_data.csv`, `rts_trips_clean.csv`

**MSTM zone system, zone polygons and peak car skim (Maryland Statewide Transportation Model)** — Maryland Department of Transportation State Highway Administration (MDOT SHA), provided to the project
- Where: <https://roads.maryland.gov>
- How: Not a public download: the 1,588-zone MSTM system (zoneSystem.csv), zone polygons (SMZ_RMZ_02152011inMSTM_EPSG26918.shp) and the AM-peak auto travel-time skim were provided by MDOT. Request them from MDOT SHA (MSTM / travel forecasting).
- Put in: `00_shared_data/raw/mstm/`
- Files: `zoneSystem.csv`, `SMZ_RMZ_02152011inMSTM_EPSG26918.shp/.shx/.dbf/.prj`, `auto_peak_time.csv.gz`

**LEHD LODES 8, Workplace Area Characteristics 2016 and geography crosswalk, Maryland** — U.S. Census Bureau, LEHD
- Where: <https://lehd.ces.census.gov/data/lodes/LODES8/md/wac/md_wac_S000_JT00_2016.csv.gz>
- How: Download md_wac_S000_JT00_2016.csv.gz and https://lehd.ces.census.gov/data/lodes/LODES8/md/md_xwalk.csv.gz (other states: .../LODES8/<st>/wac/<st>_wac_S000_JT00_<year>.csv.gz). Workplace zone attributes (high-earning share CE03/C000) use the 2017 WAC files of md, dc, va, pa, de, wv: https://lehd.ces.census.gov/data/lodes/LODES8/<st>/wac/<st>_wac_S000_JT00_2017.csv.gz (project: data/raw/lodes_wac/ and data/raw/lodes_wac_2017_extra/).
- Put in: `00_shared_data/raw/lodes_wac/`
- Files: `md_wac_S000_JT00_2016.csv.gz`, `md_xwalk.csv.gz`, `<st>_wac_S000_JT00_2017.csv.gz (st = md, dc, va, pa, de, wv)`

**BLS JOLTS job-openings rate, Maryland** — U.S. Bureau of Labor Statistics
- Where: <https://api.bls.gov/publicAPI/v1/timeseries/data/JTS000000240000000JOR>
- How: BLS public API v1, series JTS000000240000000JOR (Maryland) and JTS000000000000000JOR (U.S.), annual mean of monthly SA values.
- Put in: `00_shared_data/raw/jolts/`
- Files: `jolts_bls_api.json`

**Census 2010 Redistricting (PL 94-171) Maryland file and TIGER 2010 blocks and tracts** — U.S. Census Bureau
- Where: <https://www2.census.gov/census_2010/01-Redistricting_File--PL_94-171/Maryland/>
- How: md2010.pl.zip (geo file and segment 2, housing units H1) and TIGER/Line 2010 tl_2010_24_tabblock10.zip; tracts tl_2010_<st>_tract10.zip for st = 10, 11, 24, 42, 51, 54 (https://www2.census.gov/geo/tiger/TIGER2010/TRACT/2010/).
- Put in: `00_shared_data/raw/census2010/`
- Files: `md2010.pl.zip`, `tl_2010_24_tabblock10.zip`, `tl_2010_<st>_tract10.zip`

In [ ]:
# files of input/data/raw (the model's own extracts; shared sources are in ../00_shared_data)
raw = os.path.join(FOLDER, "input", "data", "raw")
display(nt.files_table(raw))
for f in sorted(glob.glob(os.path.join(raw, "*")))[:6]:
    head, n = nt.preview(f)
    if head is not None:
        print(f"\n{os.path.basename(f)}" + (f"  ({n:,} rows)" if n else ""))
        display(head)

In [ ]:
print(open(os.path.join(raw, "README.md")).read() if os.path.exists(os.path.join(raw, "README.md")) else "no README in input/data/raw")

## 2. Data preparation

Upstream stages (outputs provided in `00_shared_data` and `input/data/raw`; run them only for a full rebuild from the raw data, commands from the model folder):

- **Step 1.** Synthetic population, VAE generation (upstream project stage; output provided (redo only for a rebuild from raw data; writes into the project folders: back them up first)); the generative model is in the Updated VAE project outside Validation_2026  
  `cd ../../../../Updated VAE` → `see the README of that project (steps 01-04: preprocessing of ACS PUMS, training, generation)`
- **Step 2.** Synthetic population 2016, corrections, jobs, workplaces, dwellings, vacancies (upstream project stage; output provided (redo only for a rebuild from raw data; writes into the project folders: back them up first))  
  `cd ../../../1_vae` → `/opt/homebrew/anaconda3/bin/python3 zone_weights_2020.py && /opt/homebrew/anaconda3/bin/python3 zone_weights_2016.py && /opt/homebrew/anaconda3/bin/python3 rebalance_population.py && /opt/homebrew/anaconda3/bin/python3 employment_2016_lodes.py && /opt/homebrew/anaconda3/bin/python3 workplace_balanced.py && /opt/homebrew/anaconda3/bin/python3 export_silo_base.py && /opt/homebrew/anaconda3/bin/python3 zone_vacancy_2016.py && /opt/homebrew/anaconda3/bin/python3 market_vacancy_2016.py`
- **Step 3.** SILO base-year microdata (upstream project stage; output provided (redo only for a rebuild from raw data; writes into the project folders: back them up first))  
  `cd ../../../2_silo/input_build` → `/opt/homebrew/anaconda3/bin/python3 build_microdata.py && /opt/homebrew/anaconda3/bin/python3 build_base_v2.py`
- **Step 4.** Accessibility logsums (car-based, original activity-model destination models) (upstream project stage; output provided (redo only for a rebuild from raw data; writes into the project folders: back them up first))  
  `cd ../../..` → `/opt/homebrew/anaconda3/bin/python3 pipeline/accessibility.py --year 2016`
- **Step 5.** Workplace data of the original model: zone jobs, RTS workers with home and work zones, zone attributes (upstream project stage; output provided (redo only for a rebuild from raw data; writes into the project folders: back them up first))  
  `cd ../../../2_silo/jobchoice` → `/opt/homebrew/anaconda3/bin/python3 taz_crosswalk.py && /opt/homebrew/anaconda3/bin/python3 s01_data.py 2016 && /opt/homebrew/anaconda3/bin/python3 rts_11_data_v2.py && /opt/homebrew/anaconda3/bin/python3 rts_20_zone_attrs.py`

Package steps, run below when `RUN_DATA_PREP = True`: they rebuild `input/data/estimation` and `input/data/holdout` with the original random draws and stop if the log-likelihood of the original run is not reproduced; `check_against_provided.py` must end with *ALL TABLES MATCH*.

**Step 6.** data.npz (package copy; overwrites input/data/raw/data.npz) (about 0.5 min)

In [ ]:
if RUN_DATA_PREP:
    nt.run('~/.venvs/biogeme/bin/python w01_data.py', cwd=os.path.join(FOLDER, 'input/code'))
else:
    print("skipped (RUN_DATA_PREP = False): the provided tables are used")

**Step 7.** input_check, part 1: keep a copy of the provided estimation and hold-out tables (about 0.1 min)

In [ ]:
if RUN_DATA_PREP:
    nt.run('rm -rf provided_estimation provided_holdout && cp -R estimation provided_estimation && cp -R holdout provided_holdout', cwd=os.path.join(FOLDER, 'input/data'))
else:
    print("skipped (RUN_DATA_PREP = False): the provided tables are used")

**Step 8.** Build the estimation and hold-out tables from the inputs (same calls and random draws as the original run; asserts that the LL at the original estimates is reproduced) (about 0.2 min)

In [ ]:
if RUN_DATA_PREP:
    nt.run('~/.venvs/biogeme/bin/python build_estimation_data.py', cwd=os.path.join(FOLDER, 'input/code'))
else:
    print("skipped (RUN_DATA_PREP = False): the provided tables are used")

**Step 9.** input_check, part 2: compare the rebuilt tables with the provided ones (rows, column sums, largest difference); must print ALL TABLES MATCH (about 0.5 min)

In [ ]:
if RUN_DATA_PREP:
    nt.run('~/.venvs/biogeme/bin/python check_against_provided.py', cwd=os.path.join(FOLDER, 'input/code'))
else:
    print("skipped (RUN_DATA_PREP = False): the provided tables are used")

## 3. Estimation data

The table exactly as used in the estimation, and its data dictionary.

In [ ]:
est = os.path.join(FOLDER, "input", "data", "estimation")
display(nt.files_table(est))
dd = pd.read_csv(os.path.join(est, "data_dictionary.csv"))
dd

In [ ]:
import estimate_workplace_choice as M
df = pd.read_parquet(M.DATA)
print(f"{df.shape[0]:,} rows (decision makers), {df.shape[1]:,} columns")
df.head()

## 4. Descriptive statistics

In [ ]:
alts = nt.alt_columns(df)
J = len(next(iter(alts.values())))
print(f"{J} alternatives per decision maker (columns <variable>_<j>)")
if "av" in alts:
    n_av = df[alts["av"]].sum(axis=1)
    print("available alternatives per decision maker:", n_av.describe().round(1).to_dict())
if "choice" in df:
    c = df["choice"].value_counts().sort_index()
    ax = (100 * c / c.sum()).plot.bar(figsize=(min(14, 3 + 0.12 * J), 3), color="#2b6cb0", width=0.8)
    ax.set_xlabel("position of the chosen alternative"); ax.set_ylabel("%"); ax.set_title("Chosen alternative (position in the set)")
    ax.set_xticks(ax.get_xticks()[::max(1, J // 20)]); plt.show()
person = [c for c in df.columns if not any(c in v for v in alts.values())]
display(nt.explanatory(df[person]).describe().T.round(3))
if "weight" in df:
    ax = df["weight"].plot.hist(bins=60, figsize=(5, 3), color="#2b6cb0"); ax.set_title("Estimation weight"); plt.show()
    print("weight: mean %.3f, min %.3f, max %.3f" % (df.weight.mean(), df.weight.min(), df.weight.max()))

## 5. Model specification

The specification is the folder's script `model/estimate_workplace_choice.py`, imported as it is. The start values are the article estimates.

In [ ]:
print(inspect.getsource(M.model))

In [ ]:
t0 = time.time()
database, ll = M.model()
used = nt.model_variables(ll)
print(f"database: {database.num_rows():,} rows; {len(used)} data columns used by the model; built in {time.time()-t0:.1f} s")

## 6. Correlation of the explanatory variables

Correlations of the variables that enter the utility, computed over the alternatives available to each decision maker
(alternative-specific variables) and over decision makers (person and household variables). |r| ≥ 0.7 and VIF ≥ 10 are
listed. Interaction terms are correlated with their main effects by construction.

In [ ]:
stems = sorted({c.rsplit("_", 1)[0] for c in used if c.rsplit("_", 1)[0] in alts and c.rsplit("_", 1)[-1].isdigit()}
               - {"av", "chosen_set", "zone", "stype"})
long = nt.explanatory(nt.stack_alternatives(df, stems))
print(f"{len(stems)} alternative-specific variables, {len(long):,} available alternatives")
Ca = nt.corr_heatmap(long, "Alternative-specific variables (available alternatives)")
display(nt.high_pairs(Ca)); display(nt.vif(long))
plain = [c for c in used if c in df.columns and c not in sum(alts.values(), []) and c not in ("choice", "weight")]
if len(plain) > 1:
    Xp = nt.explanatory(df[plain]); Cp = nt.corr_heatmap(Xp, "Decision-maker variables"); display(nt.high_pairs(Cp))

## 7. Log-likelihood at the article coefficients

No estimation: the log-likelihood at the start values (the article estimates). Target: article -21334.3,
Biogeme -21334.2842 (N = 9707, K = 36).

In [ ]:
if RUN_CHECK:
    p = Parameters(); p.set_value("generate_html", False); p.set_value("generate_yaml", False)
    here = os.getcwd(); os.chdir(__import__("tempfile").mkdtemp())
    t0 = time.time()
    b = bio.BIOGEME(database, ll, parameters=p); b.model_name = NAME + "_check"
    ll0 = b.calculate_init_likelihood()
    os.chdir(here)
    print(f"N {database.num_rows():,}   LL at the article coefficients: {ll0:,.4f}   (target -21334.2842)   {time.time()-t0:.0f} s")
else:
    print("skipped (RUN_CHECK = False)")

## 8. Biogeme estimation

The same call as `model/estimate_workplace_choice.py`: Biogeme with its defaults (read from `output/biogeme.toml`), start values = article estimates.
Report written to `output/workplace_choice.html`, `.yaml`, and `output/workplace_choice_estimates.csv`.

In [ ]:
if RUN_ESTIMATION:
    here = os.getcwd(); os.chdir(OUT)
    t0 = time.time()
    b = bio.BIOGEME(database, ll); b.model_name = NAME
    results = b.estimate()
    os.chdir(here)
    print(f"estimation time {time.time()-t0:.0f} s"); print(results.short_summary())
    nt.estimates_table(results)[["parameter", "coefficient", "robust_se", "t", "p"]].rename(
        columns={"coefficient": "Value", "robust_se": "Robust std err.", "t": "Robust t-stat.", "p": "Robust p-value", "parameter": "Name"}
    ).to_csv(os.path.join(OUT, f"{NAME}_estimates.csv"), index=False)
else:
    y = sorted(glob.glob(os.path.join(OUT, f"{NAME}*.yaml")), key=os.path.getmtime)
    results = EstimationResults.from_yaml_file(y[-1]) if y else None
    print(f"RUN_ESTIMATION = False: read {os.path.basename(y[-1])}" if y else "RUN_ESTIMATION = False and no result in output/: sections 9-10 are skipped")

## 9. Results

### Estimates in the article format
Coefficient (stars: *** p < 0.001, ** p < 0.01, * p < 0.05), robust s.e., p.

In [ ]:
if results is not None:
    est_t = nt.estimates_table(results)
    display(nt.article_format(est_t))

### Goodness of fit

In [ ]:
if results is not None:
    display(nt.fit_statistics(results))

### Comparison with the article (`model/reference_estimates.csv`)

In [ ]:
if results is not None:
    REF = os.path.join(FOLDER, "model", "reference_estimates.csv")
    cmp = nt.compare(est_t, REF)
    cmp.to_csv(os.path.join(OUT, "comparison_with_article_notebook.csv"), index=False)
    print(f"{int(cmp.matches_article.sum())} of {len(cmp)} parameters match the article (coefficient and robust s.e. within the 4th-decimal rounding); "
          f"largest |difference|: coefficient {cmp.diff_coef.abs().max():.5f}, s.e. {cmp.diff_se.abs().max():.5f}")
    display(cmp.round({"coefficient": 4, "robust_se": 4, "p": 4, "diff_coef": 5, "diff_se": 5}))
    stats = nt.article_statistics(REF)
    if len(stats):
        stats["Biogeme"] = stats["statistic"].map({"LL(beta)": results.final_loglikelihood, "N": database.num_rows() if "database" in dir() else None,
                                                   "K": results.number_of_free_parameters})
        display(stats)
    nt.plot_against_article(cmp, RUN.get("model", NAME))

### Correlation of the estimates
Robust correlation matrix of the estimated parameters; pairs with |r| ≥ 0.7 listed.

In [ ]:
if results is not None:
    display(nt.parameter_correlation(results, "Robust correlation of the estimates"))

## 10. Hold-out sample

The hold-out of this model is stored in another layout (the full choice set; see `input/data/holdout` and `model/README.md`); its log-likelihood is computed by the project's validation scripts, not in this notebook.